In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

merged = pd.read_csv('../data/all_merged.csv')
promotion = pd.read_csv('../data/promotion_df.csv')
normal = pd.read_csv('../data/normal_flow_df.csv')
final = pd.read_csv('../data/promotion_df_final.csv')

---

In [3]:
display(final.head())
final.shape

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,flow_type,amount_from_received,amount_from_viewed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,72000.0,8.57,1,1,1,0,1,0,8.57,0.00
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,72000.0,14.11,1,1,1,0,1,0,14.11,0.00
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,72000.0,10.27,1,0,1,0,1,2,10.27,0.00
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,57000.0,11.93,1,1,1,1,1,1,11.93,11.93
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,57000.0,22.05,1,1,1,1,1,1,22.05,22.05


(76277, 27)

---

 전체 수신자 (100%)<br>
  └── 열람율: 전체 received 중에 viewed 된 것(정상 + 비정상)<br>
  │ &nbsp;&nbsp;&nbsp;└── 정상 열람율: received → viewed가 순서대로 시행된 것<br>
  │              &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└──정상 전환율: 정상 열람(received → viewed)된 것 중에 completed된 비율<br>
  └── 정상 완료율: 전체 received 중 received → viewed → completed가 시행된 비율<br>
  └── 비정 완료율: 전체 received 중 received → completed → viewed가 시행된 비율<br>
  └── 바로완료율: 전체 received 중 received → completed가 시행된 비율<br>
  └── 완료율: 전체 received 중 completed가 된 것(정상 + 비정상)<br>

| flow_type | 흐름 설명                     | 비고            |
|-----------|------------------------------|-----------------|
| 0         | 받기 → 완료 → 보기            | 비정상 흐름      |
| 1         | 받기 → 보기 → 완료            | 정상 흐름        |
| 2         | 받기 → NaN -> 완료                   | 바로 완료   |
| 3         | 받기 → 보기 -> NaN                   | 완료 없이 보기   |
| 4         | 받기 -> NaN -> NaN                        | 반응 없음        |

<br>

| 지표 | 분자 | 분모 |
|---|---|---|
| 열람율 | `flow_type.isin([0,1,2,3,4])`의 is_viewed | `flow_type.isin([0,1,2,3,4])`의 count |
| 정상 열람율 | `flow_type.isin([1,3])`의 is_viewed | `flow_type.isin([0,1,2,3,4])`의 count |
| 정상 전환율 | `flow_type == 1`의 count | `flow_type.isin([1,3])`의 count |
| 정상 완료율 | `flow_type == 1`의 count | `flow_type.isin([0,1,2,3,4])`의 count |
| 비정상 완료율 | `flow_type == 0`의 count | `flow_type.isin([0,1,2,3,4])`의 count |
| 바로완료율 | `flow_type == 2`의 count | `flow_type.isin([0,1,2,3,4])`의 count |
| 완료율 | `flow_type.isin([0,1,2,3,4])`의 is_completed | `flow_type.isin([0,1,2,3,4])`의 count |

---

# 쿠폰

In [4]:
# bogo, discount만 필터링
bd_df = final[final['offer_type'].isin(['bogo', 'discount'])].copy()

In [5]:
bd_df.shape

(61042, 27)

In [6]:
normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_funnel_dict(total, viewed, normal, invalid, completed_only):
    normal_viewed = total[total['flow_type'].isin([1, 3])]['is_viewed'].sum()
    
    return {
        'total_received': len(total),
        'total_viewed': total['is_viewed'].sum(),
        'total_completed': total['is_completed'].sum(),
        'normal_flow': len(normal),
        '열람율(%)': round(total['is_viewed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
        '정상 열람율(%)': round(normal_viewed / len(total) * 100, 2) if len(total) > 0 else 0,
        '정상 전환율(%)': round(len(normal) / len(viewed) * 100, 2) if len(viewed) > 0 else 0,
        '정상 완료율(%)': round(len(normal) / len(total) * 100, 2) if len(total) > 0 else 0,
        '비정상 완료율(%)': round(len(invalid) / len(total) * 100, 2) if len(total) > 0 else 0,
        '바로완료율(%)': round(len(completed_only) / len(total) * 100, 2) if len(total) > 0 else 0,
        '완료율(%)': round(total['is_completed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
    }

# 전체
overall_funnel = pd.DataFrame([{
    'group': 'overall',
    **get_funnel_dict(bd_df, viewed_bd_df, normal_bd_df, invalid_bd_df, completed_only_bd_df)
}])
display(overall_funnel)

# offer_type
type_funnel_list = []
for offer_type, group in bd_df.groupby('offer_type'):
    viewed = viewed_bd_df[viewed_bd_df['offer_type'] == offer_type]
    normal = normal_bd_df[normal_bd_df['offer_type'] == offer_type]
    invalid = invalid_bd_df[invalid_bd_df['offer_type'] == offer_type]
    completed_only = completed_only_bd_df[completed_only_bd_df['offer_type'] == offer_type]
    type_funnel_list.append({
        'offer_type': offer_type,
        **get_funnel_dict(group, viewed, normal, invalid, completed_only)
    })

type_funnel = pd.DataFrame(type_funnel_list)
display(type_funnel)

# offer_id
id_funnel_list = []
for offer_id, group in bd_df.groupby('offer_id'):
    viewed = viewed_bd_df[viewed_bd_df['offer_id'] == offer_id]
    normal = normal_bd_df[normal_bd_df['offer_id'] == offer_id]
    invalid = invalid_bd_df[invalid_bd_df['offer_id'] == offer_id]
    completed_only = completed_only_bd_df[completed_only_bd_df['offer_id'] == offer_id]
    id_funnel_list.append({
        'offer_id': offer_id,
        **get_funnel_dict(group, viewed, normal, invalid, completed_only)
    })

id_funnel = pd.DataFrame(id_funnel_list)
display(id_funnel)

,group,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
0,overall,61042,46894,33101,23267,76.82,69.93,54.5,38.12,6.89,9.22,54.23


,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
0,bogo,30499,25449,15501,10941,83.44,75.85,47.29,35.87,7.59,7.36,50.82
1,discount,30543,21445,17600,12326,70.21,64.02,63.03,40.36,6.19,11.08,57.62


,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
0,bogo_10_10_5,7593,7298,3301,2739,96.11,89.90,40.13,36.07,6.22,1.19,43.47
1,bogo_10_10_7,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47
2,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29
3,bogo_5_5_7,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
4,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
5,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82
6,discount_20_5_10,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
7,discount_7_3_7,7646,7337,5102,4348,95.96,88.14,64.52,56.87,7.82,2.04,66.73


In [7]:
# 정상 흐름만
normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()

# bogo / discount 분리
valid_bogo_transactions = normal_bd_df[normal_bd_df['offer_type'] == 'bogo'].copy()
valid_discount_transactions = normal_bd_df[normal_bd_df['offer_type'] == 'discount'].copy()

print(f"bogo 유효 거래 수: {len(valid_bogo_transactions)}")
print(f"discount 유효 거래 수: {len(valid_discount_transactions)}")


bogo 유효 거래 수: 10941
discount 유효 거래 수: 12326


In [8]:
# bogo 건당 매출
bogo_instance_revenue = valid_bogo_transactions.groupby(['person', 'offer_id', 'offer_cycle']).agg(
    total_amount=('amount', 'sum')
).reset_index()

# bogo 최종 지표
bogo_performance = bogo_instance_revenue.groupby('offer_id').agg(
    conversion_count=('person', 'count'),
    total_revenue=('total_amount', 'sum')
).reset_index().sort_values(by='total_revenue', ascending=False)

display(bogo_performance)

# discount 건당 매출
discount_instance_revenue = valid_discount_transactions.groupby(['person', 'offer_id', 'offer_cycle']).agg(
    total_amount=('amount', 'sum')
).reset_index()

# discount 최종 지표
discount_performance = discount_instance_revenue.groupby('offer_id').agg(
    conversion_count=('person', 'count'),
    total_revenue=('total_amount', 'sum')
).reset_index().sort_values(by='total_revenue', ascending=False)

display(discount_performance)

# 프로모션 타입별 건당 매출
type_instance_revenue = normal_bd_df.groupby(['person', 'offer_type', 'offer_id', 'offer_cycle']).agg(
    total_amount=('amount', 'sum')
).reset_index()

# 프로모션 타입별 최종 지표
type_performance = type_instance_revenue.groupby('offer_type').agg(
    conversion_count=('person', 'count'),
    total_revenue=('total_amount', 'sum')
).reset_index().sort_values(by='total_revenue', ascending=False)

display(type_performance)


,offer_id,conversion_count,total_revenue
2,bogo_5_5_5,3514,70093.72
0,bogo_10_10_5,2739,65289.48
1,bogo_10_10_7,2582,61508.30
3,bogo_5_5_7,2106,37422.05


,offer_id,conversion_count,total_revenue
0,discount_10_2_10,4576,82824.93
3,discount_7_3_7,4348,74016.92
1,discount_10_2_7,2102,42263.73
2,discount_20_5_10,1300,34579.64


,offer_type,conversion_count,total_revenue
0,bogo,10941,234313.55
1,discount,12326,233685.22


- `정상 흐름: 받기 -> 보기 -> 완료`
## 1. 쿠폰 전체 요약
- 전체 61,042명이 쿠폰을 받았고, 그 중 76.82%가 열람했다.
    - 이 중 정상 흐름으로 열람한 고객은 69.93%이다.
- 열람한 고객 중 정상 흐름으로 완료까지 이어진 정상 전환율은 54.5%로, 열람자의 절반 가량만 최종 완료에 도달했다.
- 전체 기준 완료율은 54.23%이며, 정상 흐름으로 완료한 비중은 38.12%이다.

## 2. 쿠폰 타입별 요약
- `bogo`와 `discount` 모두 약 30,500건으로 수신 수는 거의 동일하다.
- 열람율은 `bogo`(83.44%)가 `discount`(70.32%)보다 13%p 높아 bogo 쿠폰에 대한 초기 관심도가 더 높다.
- 정상 전환율은 `discount`(63.03%)가 `bogo`(47.29%)보다 높으며, 정상 흐름 비중 역시 `discount`(40.36%)가 `bogo`(35.87%)보다 높다.
- `bogo`는 열람은 잘 되지만 완료로 이어지지 않는 경향이 있고, `discount`는 열람율은 낮지만 한번 보면 완료까지 이어지는 경향이 있다.

## 3. 쿠폰별 요약
### 3-1. bogo
- `bogo_10_10_5`가 열람율 96.11%로 가장 높지만, 완료율은 43.47%로 `bogo` 중 가장 낮다. 난이도(10)가 높아 열람 후 완료로 이어지지 않는 경향이 있다.
- `bogo_5_5_5`는 열람율(95.95%)과 완료율(56.29%) 모두 균형잡혀 있으며,
  정상 완료율(46.41%)도 `bogo` 중 가장 높아 가장 안정적인 성과를 보인다.
- `bogo_5_5_7`은 열람율이 54.33%로 `bogo` 중 가장 낮지만, 바로완료율이 20.82%로 매우 높아 쿠폰을 보지 않고 바로 완료하는 고객이 많다.

### 3-2. discount
- `discount_10_2_10`은 열람율(96.45%), 완료율(68.84%), 정상 완료율(60.23%) 모두 전체 8개 오퍼 중 가장 높아 가장 효과적인 오퍼이다.
- `discount_7_3_7`도 열람율(95.96%), 완료율(66.73%)로 높은 성과를 보이며 `discount_10_2_10`과 함께 `discount` 타입의 우수 오퍼로 꼽힌다.
- `discount_20_5_10`은 열람율이 34.73%로 전체 중 가장 낮고, 바로완료율(23.07%)이 높아 난이도(20)가 너무 높아 열람 자체를 포기하는 경향이 있다.
- `discount_10_2_7`은 열람율(53.96%)과 완료율(51.82%) 모두 중간 수준이나 바로완료율(17.27%)이 높아 `discount_20_5_10`과 유사한 패턴을 보인다.


---

# 채널별

In [9]:
channels = ['web', 'email', 'mobile', 'social']

normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_channel_funnel_dict(total, viewed, normal, invalid, completed_only):
    normal_viewed = total[total['flow_type'].isin([1, 3])]['is_viewed'].sum()
    return {
        'total_received': len(total),
        'total_viewed': total['is_viewed'].sum(),
        'total_completed': total['is_completed'].sum(),
        'normal_flow': len(normal),
        '열람율(%)': round(total['is_viewed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
        '정상 열람율(%)': round(normal_viewed / len(total) * 100, 2) if len(total) > 0 else 0,
        '정상 전환율(%)': round(len(normal) / len(viewed) * 100, 2) if len(viewed) > 0 else 0,
        '정상 완료율(%)': round(len(normal) / len(total) * 100, 2) if len(total) > 0 else 0,
        '비정상 완료율(%)': round(len(invalid) / len(total) * 100, 2) if len(total) > 0 else 0,
        '바로완료율(%)': round(len(completed_only) / len(total) * 100, 2) if len(total) > 0 else 0,
        '완료율(%)': round(total['is_completed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
    }

# 채널 전체
channel_funnel_list = []
for channel in channels:
    total = bd_df[bd_df[channel] == 1]
    viewed = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal = normal_bd_df[normal_bd_df[channel] == 1]
    invalid = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    channel_funnel_list.append({
        'channel': channel,
        **get_channel_funnel_dict(total, viewed, normal, invalid, completed_only)
    })

channel_funnel = pd.DataFrame(channel_funnel_list)
display(channel_funnel)

# 채널 x offer_type
channel_type_list = []
for channel in channels:
    total_temp = bd_df[bd_df[channel] == 1]
    viewed_temp = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal_temp = normal_bd_df[normal_bd_df[channel] == 1]
    invalid_temp = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only_temp = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    for offer_type in total_temp['offer_type'].unique():
        total_group = total_temp[total_temp['offer_type'] == offer_type]
        viewed_group = viewed_temp[viewed_temp['offer_type'] == offer_type]
        normal_group = normal_temp[normal_temp['offer_type'] == offer_type]
        invalid_group = invalid_temp[invalid_temp['offer_type'] == offer_type]
        completed_only_group = completed_only_temp[completed_only_temp['offer_type'] == offer_type]
        channel_type_list.append({
            'channel': channel,
            'offer_type': offer_type,
            **get_channel_funnel_dict(total_group, viewed_group, normal_group, invalid_group, completed_only_group)
        })

channel_type_funnel = pd.DataFrame(channel_type_list)
display(channel_type_funnel)

# 채널 x offer_id
channel_id_list = []
for channel in channels:
    total_temp = bd_df[bd_df[channel] == 1]
    viewed_temp = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal_temp = normal_bd_df[normal_bd_df[channel] == 1]
    invalid_temp = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only_temp = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    for offer_id in total_temp['offer_id'].unique():
        total_group = total_temp[total_temp['offer_id'] == offer_id]
        viewed_group = viewed_temp[viewed_temp['offer_id'] == offer_id]
        normal_group = normal_temp[normal_temp['offer_id'] == offer_id]
        invalid_group = invalid_temp[invalid_temp['offer_id'] == offer_id]
        completed_only_group = completed_only_temp[completed_only_temp['offer_id'] == offer_id]
        channel_id_list.append({
            'channel': channel,
            'offer_id': offer_id,
            **get_channel_funnel_dict(total_group, viewed_group, normal_group, invalid_group, completed_only_group)
        })

channel_id_funnel = pd.DataFrame(channel_id_list)
display(channel_id_funnel)

,channel,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
0,web,53384,40178,29466,20685,75.26,68.56,56.52,38.75,6.70,9.74,55.20
1,email,61042,46894,33101,23267,76.82,69.93,54.50,38.12,6.89,9.22,54.23
2,mobile,53374,44231,29788,21967,82.87,75.45,54.55,41.16,7.42,7.23,55.81
3,social,38065,35942,21530,17759,94.42,87.00,53.63,46.65,7.43,2.48,56.56


,channel,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
0,web,bogo,22841,18733,11866,8359,82.01,74.62,49.04,36.60,7.39,7.96,51.95
1,web,discount,30543,21445,17600,12326,70.21,64.02,63.03,40.36,6.19,11.08,57.62
2,email,bogo,30499,25449,15501,10941,83.44,75.85,47.29,35.87,7.59,7.36,50.82
3,email,discount,30543,21445,17600,12326,70.21,64.02,63.03,40.36,6.19,11.08,57.62
4,mobile,bogo,30499,25449,15501,10941,83.44,75.85,47.29,35.87,7.59,7.36,50.82
5,mobile,discount,22875,18782,14287,11026,82.11,74.91,64.34,48.20,7.20,7.06,62.46
6,social,bogo,22822,21278,11198,8835,93.23,85.72,45.16,38.71,7.52,2.83,49.07
7,social,discount,15243,14664,10332,8924,96.20,88.91,65.85,58.54,7.29,1.95,67.78


,channel,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
0,web,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29
1,web,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
2,web,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82
3,web,discount_7_3_7,7646,7337,5102,4348,95.96,88.14,64.52,56.87,7.82,2.04,66.73
4,web,bogo_5_5_7,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
5,web,discount_20_5_10,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
6,web,bogo_10_10_5,7593,7298,3301,2739,96.11,89.90,40.13,36.07,6.22,1.19,43.47
7,email,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29
8,email,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
9,email,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82


## 1. 채널별 x 전체
- 수신 수는 `email`(61,042)이 가장 많고, `social`(38,065)이 가장 적다.
- 열람율은 `social`(94.42%)이 압도적으로 높고, `web`(75.26%)이 가장 낮다. `social` 채널은 쿠폰을 받으면 거의 대부분 열람하는 경향이 있다.
- 정상 전환율은 `web`(56.52%)이 가장 높고, `social`(53.63%)이 가장 낮다. 열람율이 높은 `social`이 오히려 전환율은 낮아, 열람 후 완료로 이어지지 않는 경향이 있다.
- 정상 완료율은 `social`(46.65%)이 가장 높고, `email`(38.12%)이 가장 낮다.
- 바로완료율은 `web`(9.74%)과 `email`(9.22%)이 높고, `social`(2.48%)이 매우 낮다. `web`과 `email`은 쿠폰을 보지 않고 바로 완료하는 고객 비중이 상대적으로 높다.
- 최종 완료율은 채널 간 큰 차이 없이 54~56% 수준으로 고르게 나타난다.

## 2. 채널별 x 쿠폰 타입별
### 2-1. bogo
- `bogo` 열람율은 `social`(93.23%)이 가장 높고, `web`(82.01%)이 가장 낮다.
- 정상 전환율은 `web`(49.04%)이 가장 높고, `social`(45.16%)이 가장 낮아 `social`은 `bogo` 열람 후 완료로 이어지는 비율이 가장 낮다.
- 완료율은 `web`(51.95%), `email`(50.82%), `mobile`(50.82%), `social`(49.07%) 순으로 채널 간 큰 차이 없이 50% 내외로 고르게 나타난다.

### 2-2. discount
- `discount` 열람율은 `social`(96.20%)이 압도적으로 높고, `email`(70.21%)이 가장 낮다.
- 정상 전환율은 `mobile`(64.34%)과 `social`(65.85%)이 높고, `email`(63.03%)과 `web`(63.03%)이 낮다.
- 완료율은 `social`(67.78%)이 가장 높고, `email`(57.62%)이 가장 낮다.

### 채널 x 쿠폰 타입 종합
- 모든 채널에서 `discount`가 `bogo`보다 완료율이 높다.
- `social` 채널은 `bogo`보다 `discount`에서 특히 강한 성과를 보이며, 열람율(96.20%)과 완료율(67.78%) 모두 전체 조합 중 가장 높다.
- `web`과 `email`은 `bogo`/`discount` 간 완료율 차이가 크지 않아 채널 특성보다 오퍼 타입의 영향을 덜 받는 편이다.

## 3. 채널별 x 쿠폰별
### 3-1. 상위 성과 오퍼
- `discount_10_2_10`은 열람율(96.45%), 완료율(68.84%), 정상 완료율(60.23%) 모두 전체 조합 중 가장 높아 채널에 관계없이 가장 효과적인 오퍼이다.
- `discount_7_3_7`도 열람율(95.96%), 완료율(66.73%)로 전반적으로 높은 성과를 보인다.
- `bogo_5_5_5`는 `bogo` 중 열람율(95.95%)과 완료율(56.29%)이 가장 균형잡혀 있다.

### 3-2. 하위 성과 오퍼
- `discount_20_5_10`은 열람율(34.73%)과 정상 완료율(16.95%)이 전체 중 가장 낮으며, 바로완료율(23.07%)이 높아 난이도(20)가 너무 높아 열람 자체를 포기하는 경향이 있다.
- `bogo_10_10_5`와 `bogo_5_5_7`도 열람율 대비 완료율이 낮아 난이도가 높을수록 전환 효율이 떨어지는 패턴이 확인된다.

## 문제점
- 이 데이터는 채널이 분리된 단일 채널 기준이 아니라, 여러 채널이 조합되어 발송된 원본 데이터를 채널별로 분리해서 집계한 결과이다.
- 따라서 `web/email/mobile/social` 모든 채널에서 동일한 수치가 반복되고 있어 채널별 순수 성과 비교가 불가능한 상태이다.

---

# 채널 조합별

In [10]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)

normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group):
    normal_viewed = group[group['flow_type'].isin([1, 3])]['is_viewed'].sum()
    return {
        'total_received': len(group),
        'total_viewed': group['is_viewed'].sum(),
        'total_completed': group['is_completed'].sum(),
        'normal_flow': len(normal_group),
        '열람율(%)': round(group['is_viewed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '정상 열람율(%)': round(normal_viewed / len(group) * 100, 2) if len(group) > 0 else 0,
        '정상 전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
        '정상 완료율(%)': round(len(normal_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '비정상 완료율(%)': round(len(invalid_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '바로완료율(%)': round(len(completed_only_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '완료율(%)': round(group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
    }

# 채널 조합 전체
combo_funnel_list = []
for combo, group in bd_df.groupby('channel_combo'):
    viewed_group = viewed_bd_df[viewed_bd_df['channel_combo'] == combo]
    normal_group = normal_bd_df[normal_bd_df['channel_combo'] == combo]
    invalid_group = invalid_bd_df[invalid_bd_df['channel_combo'] == combo]
    completed_only_group = completed_only_bd_df[completed_only_bd_df['channel_combo'] == combo]
    combo_funnel_list.append({
        'channel_combo': combo,
        **get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group)
    })

combo_funnel = pd.DataFrame(combo_funnel_list).sort_values('total_received', ascending=False)
display(combo_funnel)

# 채널 조합 x offer_type
combo_type_list = []
for (combo, offer_type), group in bd_df.groupby(['channel_combo', 'offer_type']):
    viewed_group = viewed_bd_df[(viewed_bd_df['channel_combo'] == combo) & (viewed_bd_df['offer_type'] == offer_type)]
    normal_group = normal_bd_df[(normal_bd_df['channel_combo'] == combo) & (normal_bd_df['offer_type'] == offer_type)]
    invalid_group = invalid_bd_df[(invalid_bd_df['channel_combo'] == combo) & (invalid_bd_df['offer_type'] == offer_type)]
    completed_only_group = completed_only_bd_df[(completed_only_bd_df['channel_combo'] == combo) & (completed_only_bd_df['offer_type'] == offer_type)]
    combo_type_list.append({
        'channel_combo': combo,
        'offer_type': offer_type,
        **get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group)
    })

combo_type_funnel = pd.DataFrame(combo_type_list).sort_values('total_received', ascending=False)
display(combo_type_funnel)

# 채널 조합 x offer_id
combo_id_list = []
for (combo, offer_id), group in bd_df.groupby(['channel_combo', 'offer_id']):
    viewed_group = viewed_bd_df[(viewed_bd_df['channel_combo'] == combo) & (viewed_bd_df['offer_id'] == offer_id)]
    normal_group = normal_bd_df[(normal_bd_df['channel_combo'] == combo) & (normal_bd_df['offer_id'] == offer_id)]
    invalid_group = invalid_bd_df[(invalid_bd_df['channel_combo'] == combo) & (invalid_bd_df['offer_id'] == offer_id)]
    completed_only_group = completed_only_bd_df[(completed_only_bd_df['channel_combo'] == combo) & (completed_only_bd_df['offer_id'] == offer_id)]
    combo_id_list.append({
        'channel_combo': combo,
        'offer_id': offer_id,
        **get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group)
    })

combo_id_funnel = pd.DataFrame(combo_id_list).sort_values('total_received', ascending=False)
display(combo_id_funnel)

,channel_combo,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
3,web+email+mobile+social,30407,29226,17895,15177,96.12,88.88,56.16,49.91,7.24,1.70,58.85
2,web+email+mobile,15309,8289,8258,4208,54.14,46.74,58.81,27.49,7.41,19.05,53.94
1,web+email,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
0,email+mobile+social,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47


,channel_combo,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
5,web+email+mobile+social,discount,15243,14664,10332,8924,96.20,88.91,65.85,58.54,7.29,1.95,67.78
4,web+email+mobile+social,bogo,15164,14562,7563,6253,96.03,88.84,46.41,41.24,7.19,1.45,49.87
2,web+email+mobile,bogo,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
1,web+email,discount,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
0,email+mobile+social,bogo,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47
3,web+email+mobile,discount,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82


,channel_combo,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상 열람율(%),정상 전환율(%),정상 완료율(%),비정상 완료율(%),바로완료율(%),완료율(%)
2,web+email+mobile,bogo_5_5_7,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
1,web+email,discount_20_5_10,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
0,email+mobile+social,bogo_10_10_7,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47
7,web+email+mobile+social,discount_7_3_7,7646,7337,5102,4348,95.96,88.14,64.52,56.87,7.82,2.04,66.73
3,web+email+mobile,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82
6,web+email+mobile+social,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
4,web+email+mobile+social,bogo_10_10_5,7593,7298,3301,2739,96.11,89.90,40.13,36.07,6.22,1.19,43.47
5,web+email+mobile+social,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29


## 1. 채널 조합별 전체 요약
- 총 4가지 채널 조합이 존재하며, 모든 채널을 동시에 활용한 `web+email+mobile+social` 조합이 30,407건으로 가장 많이 발송되었다.

### 1-1. 성과
- `web+email+mobile+social`은 열람율(96.12%), 완료율(58.85%), 정상 완료율(49.91%) 모두 4개 조합 중 가장 높아 많은 채널을 동시에 활용할수록 전반적인 성과가 높아지는 경향이 있다.
- `email+mobile+social`은 열람율(87.70%)과 완료율(47.47%)이 `web+email+mobile`보다 높아, `web` 채널 없이도 준수한 성과를 보인다.
- `web+email`은 열람율(34.73%)과 완료율(43.21%)이 가장 낮으며, 바로완료율(23.07%)이 높아 쿠폰을 열람하지 않고 바로 완료하는 고객 비중이 가장 높다.
- `web+email+mobile`은 열람율(54.14%)과 정상 완료율(27.49%)이 낮고, 바로완료율(19.05%)도 높아 `web+email` 조합과 유사한 패턴을 보인다.

### 1-2. 종합
- `web` 채널이 포함된 조합(`web+email`, `web+email+mobile`)에서 열람율이 낮고 바로완료율이 높은 패턴이 공통적으로 나타난다.
- 반면 `social` 채널이 포함된 조합에서 열람율이 높게 나타나 `social` 채널이 열람을 유도하는 데 핵심적인 역할을 하는 것으로 보인다.

## 2. 채널 조합별 x 쿠폰 타입별 요약

### 2-1. web+email+mobile+social (전채널)
- `discount`(96.20%, 완료율 67.78%)와 `bogo`(96.03%, 완료율 49.87%) 모두 열람율은 거의 동일하지만 완료율은 `discount`가 약 18%p 높다.
- 4개 채널을 모두 활용할 때 `discount`가 압도적으로 효과적이다.

### 2-2. 부분 채널 조합
- `web+email+mobile`의 `bogo`는 열람율(54.33%)이 낮지만 완료율(56.05%)이 `web+email+mobile+social` `bogo`(49.87%)보다 오히려 높다.
- `web+email discount`는 열람율(34.73%)과 완료율(43.21%)이 전체 중 가장 낮으며, 바로완료율(23.07%)이 높아 쿠폰 열람 없이 완료되는 비중이 크다.
- `email+mobile+social`의 `bogo`는 열람율(87.70%)에 비해 완료율(47.47%)이 낮아 열람 후 완료로 이어지는 전환이 약하다.
- `web+email+mobile`의 `discount`는 열람율(53.96%)과 완료율(51.82%) 모두 중간 수준이나 바로완료율(17.27%)이 높아 `web+email` 조합과 유사한 패턴을 보인다.

### 2-3. 종합
- `social` 채널 포함 여부가 열람율에 결정적인 영향을 미치며, `social`이 빠지면 열람율이 크게 낮아지는 경향이 있다.
- `discount`는 모든 채널 조합에서 `bogo`보다 완료율이 높으며, 특히 전채널(`web+email+mobile+social`) 조합에서 가장 큰 차이를 보인다.

## 3. 채널 조합별 x 쿠폰별 요약

### 3-1. web+email+mobile+social (전채널)
- `discount_10_2_10`이 열람율(96.45%), 완료율(68.84%), 정상 완료율(60.23%) 모두 전체 조합 중 가장 높아 가장 효과적인 조합이다.
- `discount_7_3_7`도 열람율(95.96%), 완료율(66.73%)로 준수한 성과를 보인다.
- `bogo_10_10_5`는 열람율(96.11%)이 높지만 완료율(43.47%)이 낮아 난이도(10)가 전환을 방해하는 것으로 보인다.
- `bogo_5_5_5`는 `bogo` 중 완료율(56.29%)과 정상 완료율(46.41%)이 가장 높아 전채널 `bogo` 중 가장 효율적이다.

### 3-2. 부분 채널 조합
- `web+email+mobile`의 `bogo_5_5_7`은 열람율(54.33%)이 낮지만 바로완료율(20.82%)이 높아 쿠폰을 보지 않고 완료하는 비중이 크다.
- `web+email`의 `discount_20_5_10`은 열람율(34.73%)과 완료율(43.21%)이 전체 중 가장 낮으며, 난이도(20)가 너무 높아 열람 자체를 포기하는 경향이 있다.
- `email+mobile+social`의 `bogo_10_10_7`은 열람율(87.70%) 대비 완료율(47.47%)이 낮아 난이도(10)가 전환을 방해하고 있다.
- `web+email+mobile`의 `discount_10_2_7`은 열람율(53.96%)과 완료율(51.82%) 모두 중간 수준이나 바로완료율(17.27%)이 높다.

### 3-3. 종합
- 전채널(`web+email+mobile+social`) 조합에서만 `discount`와 `bogo` 모두 고루 분포되어 있으며, 부분 채널 조합은 특정 오퍼에만 편중되어 있다.
- 난이도가 낮은 오퍼(`discount_10_2_10`, `discount_7_3_7`)가 전채널 조합과 결합될 때 가장 높은 성과를 보인다.
- `social` 채널 제외 시 열람율이 크게 낮아지며, 바로완료율이 높아지는 패턴이 반복적으로 나타난다.
